# <span style="color:#0b486b">  FIT5221 - Assignment 3: Task 3, 4 </span>


## <span style="color:#0b486b">  Student Information</span>
***
Fullname: **[Yin Zhang]**  <br/>
Student ID: **[[Student ID removed]]**    <br/>
***

In [ ]:
! pip install torchinfo -i https://pypi.tuna.tsinghua.edu.cn/simple
! pip install huggingface_hub --index-url https://pypi.tuna.tsinghua.edu.cn/simple
! pip install transformers --no-deps --no-cache-dir --index-url https://pypi.tuna.tsinghua.edu.cn/simple


In [ ]:
%pip install timm

In [ ]:
# import library
import os
import random
import copy
import matplotlib.pyplot as plt

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from timm.models.layers import DropPath, Mlp, PatchEmbed
from torch.utils.data import DataLoader, random_split
from torchinfo import summary
from torchvision import datasets
from torchvision.datasets import OxfordIIITPet, wrap_dataset_for_transforms_v2
from torchvision.transforms import v2
from torchvision.transforms.functional import InterpolationMode

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print("Pytorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
set_seed(42) # For reproduciblity purpose, please do not modify the seed number.

import sys
from pathlib import Path

def _ensure_a3_dataset_mirrors_on_path():
    """Ensure a3_dataset_mirrors.py on sys.path regardless of notebook cwd."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip and hasattr(ip, "startup_dir"):
            candidates.insert(0, Path(ip.startup_dir))
    except Exception:
        pass
    for d in candidates:
        if (d / "a3_dataset_mirrors.py").is_file():
            s = str(d.resolve())
            if s not in sys.path:
                sys.path.insert(0, s)
            return d
    raise FileNotFoundError(
        "Cannot find a3_dataset_mirrors.py. Open the notebook from the FIT5221_Assignment_3 project root, "
        "or place a3_dataset_mirrors.py next to the notebook."
    )

_ensure_a3_dataset_mirrors_on_path()

import importlib
import a3_dataset_mirrors as _mirrors
importlib.reload(_mirrors)  # Avoid stale cached module in Jupyter

_mirrors.setup_china_mirrors()  # Tsinghua PyPI + HF mirror
_mirrors.print_mirror_plan("all", root=".")  # Download plan and checkpoint skip status


# Helper functions


In [ ]:
# You can change input params of this function if needed
def load_pet_data(data_dir, batch_size=64, img_height=224, img_width=224, transform=None):

    # Define transforms
    if transform is None:
        transform = v2.Compose([            
            v2.ToImage(),
            v2.ToDtype(torch.uint8, scale=True),
            v2.Resize((img_height, img_width), antialias=True, interpolation=InterpolationMode.BICUBIC),
            v2.ToDtype(torch.float32, scale=True),
        ])

    from a3_dataset_mirrors import download_oxford_iiit_pet
    # Mirrors: OpenDataLab -> hf-mirror -> official (download calls print_mirror_plan)
    download_oxford_iiit_pet(data_dir)

    train_ds = OxfordIIITPet(
        root=data_dir,
        split="trainval",
        download=False,
        transform=transform,
    )

    val_ds = OxfordIIITPet(
        root=data_dir,
        split="test",
        download=False,
        transform=transform,
    )

    print(f"Number of training samples: {len(train_ds)}")
    print(f"Number of validation samples: {len(val_ds)}")

    # Create DataLoader instances
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, 37


# Vision Transformer (ViT)


In [ ]:
def swish(x, beta):
    return x * torch.sigmoid(x * beta)


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        assert dim % num_heads == 0, "dim should be divisible by num_heads"
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = (
            self.qkv(x)
            .reshape(B, N, 3, self.num_heads, C // self.num_heads)
            .permute(2, 0, 3, 1, 4)
        )
        q, k, v = qkv.unbind(0)

        attention = (q @ k.transpose(-2, -1)) * self.scale
        attention = attention.softmax(dim=-1)
        attention = self.attn_drop(attention)

        x = (attention @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class ViTLayer(nn.Module):

    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.0,
        qkv_bias=False,
        drop=0.0,
        attn_drop=0.0,
        drop_path=0.0,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attention = Attention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        # NOTE: drop path for stochastic depth, we shall see if this is better than dropout here
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.ffn = Mlp(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop,
        )

    def forward(self, x):
        x = x + self.drop_path(self.attention(self.norm1(x)))
        x = x + self.drop_path(self.ffn(self.norm2(x)))
        return x


class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        embed_dim=384,
        depth=12,
        num_heads=6,
        mlp_ratio=4.0,
        qkv_bias=True,
        distilled=False,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.2,
        embed_layer=PatchEmbed,
    ):
        """
        Args:
            img_size (int, tuple): input image size
            patch_size (int, tuple): patch size
            in_chans (int): number of input channels
            num_classes (int): number of classes for classification head
            embed_dim (int): embedding dimension
            depth (int): depth of transformer
            num_heads (int): number of attention heads
            mlp_ratio (int): ratio of mlp hidden dim to embedding dim
            qkv_bias (bool): enable bias for qkv if True
            drop_rate (float): dropout rate
            attn_drop_rate (float): attention dropout rate
            drop_path_rate (float): stochastic depth rate
            embed_layer (nn.Module): patch embedding layer
        """
        super().__init__()
        self.num_features = self.embed_dim = (
            embed_dim  # num_features for consistency with other models
        )
        self.num_tokens = 2 if distilled else 1
        norm_layer = nn.LayerNorm
        act_layer = nn.GELU

        self.patch_embed = embed_layer(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = (
            nn.Parameter(torch.zeros(1, 1, embed_dim)) if distilled else None
        )
        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches + self.num_tokens, embed_dim)
        )
        self.pos_drop = nn.Dropout(p=drop_rate)

        self.drop_rate = drop_rate
        self.attn_drop_rate = attn_drop_rate

        dpr = [
            x.item() for x in torch.linspace(0, drop_path_rate, depth)
        ]  # stochastic depth decay rule
        self.layers = nn.Sequential(
            *[
                ViTLayer(
                    dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop_rate,
                    attn_drop=attn_drop_rate,
                    drop_path=dpr[i],
                    norm_layer=norm_layer,
                    act_layer=act_layer,
                )
                for i in range(depth)
            ]
        )
        self.norm = norm_layer(embed_dim)

        # Classifier head(s)
        self.head = nn.Linear(self.num_features, 1000)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {"pos_embed", "cls_token", "dist_token"}

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)

        if self.dist_token is None:
            x = torch.cat((cls_token, x), dim=1)
        else:
            x = torch.cat(
                (cls_token, self.dist_token.expand(x.shape[0], -1, -1), x), dim=1
            )
        x = self.pos_drop(x + self.pos_embed)

        x = self.layers(x)

        x = x[:, self.num_tokens :].mean(dim=1)

        x = self.norm(x)
        out = self.head(x)
        return out

# Task 3: Fine-tuning (a part of) the ViT Model (2 marks)


In [ ]:
small_vit = VisionTransformer()

# Load the pretrained ImageNet weight
missing, unexpected = small_vit.load_state_dict(torch.load("./pretrained_vit_small.pth", weights_only=False))

print("Missing:", missing)
print("Unexpected:", unexpected)

# Replace with a new head for transfer learning.
# Hidden dimension of 384, 37 classes for the OxfordIIITPet dataset
small_vit.head = nn.Linear(384, 37)
small_vit.to(device)


In [ ]:
batch_size = 32 # You can change this value if needed
summary(small_vit, (batch_size, 3, 224, 224), device=device)


In [ ]:
# Get the dataloaders for training
train_dataloader, val_dataloader, num_classes = load_pet_data(data_dir="./", batch_size=batch_size)

In [ ]:
# ===== Task 3, 4, Best Performance =====
# ===== Task 3 & 4 & Best Performance Model =====
import time
from pathlib import Path

import torch.optim as optim
from tqdm.auto import tqdm

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

NUM_EPOCHS_VIT = 12
LEARNING_RATE_VIT = 3e-4
WEIGHT_DECAY = 1e-4


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def build_vit_model():
    model = VisionTransformer()
    model.load_state_dict(torch.load("./pretrained_vit_small.pth", map_location="cpu"))
    model.head = nn.Linear(384, num_classes)
    return model.to(device)


def set_requires_grad(model, trainable=True):
    for p in model.parameters():
        p.requires_grad = trainable


def configure_full_finetune(model):
    set_requires_grad(model, True)
    return model


def configure_partial_finetune(model, num_last_layers=2):
    set_requires_grad(model, False)
    for p in model.head.parameters():
        p.requires_grad = True
    for layer in model.layers[-num_last_layers:]:
        for p in layer.parameters():
            p.requires_grad = True
    for p in model.norm.parameters():
        p.requires_grad = True
    return model


class LoRALinear(nn.Module):
    """LoRA adapter (Hu et al., ICLR 2022)."""

    def __init__(self, linear: nn.Linear, rank=8, alpha=16.0):
        super().__init__()
        self.linear = linear
        self.rank = rank
        self.scaling = alpha / rank
        self.lora_a = nn.Linear(linear.in_features, rank, bias=False)
        self.lora_b = nn.Linear(rank, linear.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_a.weight, a=5**0.5)
        nn.init.zeros_(self.lora_b.weight)
        for p in self.linear.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.linear(x) + self.lora_b(self.lora_a(x)) * self.scaling


def apply_lora_to_vit(model, rank=8, alpha=16.0):
    dev = next(model.parameters()).device
    set_requires_grad(model, False)
    for p in model.head.parameters():
        p.requires_grad = True
    for layer in model.layers:
        layer.attention.qkv = LoRALinear(layer.attention.qkv, rank=rank, alpha=alpha).to(dev)
    return model


def run_classification_epoch(model, loader, criterion, optimizer=None, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, leave=False, desc="train" if train else "test"):
        images, labels = images.to(device), labels.to(device)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return {"loss": total_loss / total, "accuracy": correct / total}


def train_vit(model, tag, checkpoint_name, epochs=NUM_EPOCHS_VIT):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE_VIT,
        weight_decay=WEIGHT_DECAY,
    )

    history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
    best_acc, best_state = 0.0, None

    total, trainable = count_params(model)
    print(f"[{tag}] Total params: {total:,} | Trainable params: {trainable:,}")

    for epoch in range(1, epochs + 1):
        train_stats = run_classification_epoch(model, train_dataloader, criterion, optimizer, train=True)
        test_stats = run_classification_epoch(model, val_dataloader, criterion, optimizer=None, train=False)

        history["train_loss"].append(train_stats["loss"])
        history["test_loss"].append(test_stats["loss"])
        history["train_acc"].append(train_stats["accuracy"])
        history["test_acc"].append(test_stats["accuracy"])

        print(
            f"[{tag}] Epoch {epoch}/{epochs} | "
            f"train loss={train_stats['loss']:.4f}, acc={train_stats['accuracy']:.4f} | "
            f"test loss={test_stats['loss']:.4f}, acc={test_stats['accuracy']:.4f}"
        )

        if test_stats["accuracy"] > best_acc:
            best_acc = test_stats["accuracy"]
            best_state = copy.deepcopy(model.state_dict())

    ckpt_path = CHECKPOINT_DIR / checkpoint_name
    model.load_state_dict(best_state)
    torch.save({"model_state_dict": best_state, "best_test_acc": best_acc, "history": history, "tag": tag}, ckpt_path)
    print(f"Saved {ckpt_path} (best test acc={best_acc:.4f})")
    return history, best_acc


def plot_vit_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["test_loss"], label="test")
    axes[0].set_title(f"{title} - Loss")
    axes[0].legend()
    axes[1].plot(epochs, history["train_acc"], label="train")
    axes[1].plot(epochs, history["test_acc"], label="test")
    axes[1].set_title(f"{title} - Accuracy")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def evaluate_vit_checkpoint(model_builder, checkpoint_path, strategy_name):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = model_builder()
    model.load_state_dict(ckpt["model_state_dict"])
    criterion = nn.CrossEntropyLoss()
    stats = run_classification_epoch(model, val_dataloader, criterion, train=False)
    print(f"Strategy: {strategy_name}")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Saved best test acc: {ckpt.get('best_test_acc', 'N/A')}")
    print(f"Current test acc: {stats['accuracy']:.4f}, loss: {stats['loss']:.4f}")
    return stats


import json

# ----- Task 3.1 Full fine-tuning -----
full_model = build_vit_model()
full_model = configure_full_finetune(full_model)
total, trainable = count_params(full_model)
print("Task 3 - Full fine-tuning")
print("Total parameters:", total)
print("Trainable parameters:", trainable)
assert trainable == total, "Full fine-tuning requires M == N"

_full_ckpt = CHECKPOINT_DIR / "vit_full_finetune.pth"
if _full_ckpt.exists():
    print("Found checkpoint, skipping full fine-tuning training.")
    _payload = torch.load(_full_ckpt, map_location=device, weights_only=False)
    full_history = _payload["history"]
    full_best_acc = _payload.get("best_test_acc", full_history["test_acc"][-1])
else:
    full_history, full_best_acc = train_vit(full_model, tag="full_finetune", checkpoint_name="vit_full_finetune.pth")
plot_vit_history(full_history, "Full Fine-tuning")

full_eval_model = configure_full_finetune(build_vit_model())
evaluate_vit_checkpoint(lambda: full_eval_model, CHECKPOINT_DIR / "vit_full_finetune.pth", "Full fine-tuning")


# ----- Task 3.2 Partial fine-tuning -----
partial_model = build_vit_model()
partial_model = configure_partial_finetune(partial_model, num_last_layers=2)
total, trainable = count_params(partial_model)
print("Task 3 - Partial fine-tuning (last 2 transformer blocks + head + norm)")
print("Total parameters:", total)
print("Trainable parameters:", trainable)
assert trainable < total, "Partial fine-tuning requires M < N"

_partial_ckpt = CHECKPOINT_DIR / "vit_partial_finetune.pth"
if _partial_ckpt.exists():
    print("Found checkpoint, skipping partial fine-tuning training.")
    _payload = torch.load(_partial_ckpt, map_location=device, weights_only=False)
    partial_history = _payload["history"]
    partial_best_acc = _payload.get("best_test_acc", partial_history["test_acc"][-1])
else:
    partial_history, partial_best_acc = train_vit(
        partial_model, tag="partial_finetune", checkpoint_name="vit_partial_finetune.pth"
    )
plot_vit_history(partial_history, "Partial Fine-tuning")

partial_eval_model = configure_partial_finetune(build_vit_model(), num_last_layers=2)
evaluate_vit_checkpoint(lambda: partial_eval_model, CHECKPOINT_DIR / "vit_partial_finetune.pth", "Partial fine-tuning")


# ----- Task 4: LoRA PEFT -----
lora_model = build_vit_model()
lora_model = apply_lora_to_vit(lora_model, rank=8, alpha=16.0)
total, trainable = count_params(lora_model)
print("Task 4 - LoRA PEFT")
print("Total parameters:", total)
print("Trainable parameters:", trainable)

_lora_ckpt = CHECKPOINT_DIR / "vit_lora_peft.pth"
if _lora_ckpt.exists():
    print("Found checkpoint, skipping LoRA training.")
    _payload = torch.load(_lora_ckpt, map_location=device, weights_only=False)
    lora_history = _payload["history"]
    lora_best_acc = _payload.get("best_test_acc", lora_history["test_acc"][-1])
else:
    lora_history, lora_best_acc = train_vit(lora_model, tag="lora_peft", checkpoint_name="vit_lora_peft.pth")
plot_vit_history(lora_history, "LoRA PEFT")

lora_eval_model = apply_lora_to_vit(build_vit_model(), rank=8, alpha=16.0)
evaluate_vit_checkpoint(lambda: lora_eval_model, CHECKPOINT_DIR / "vit_lora_peft.pth", "LoRA PEFT")


# ----- Best Performance Model -----
_results_json = Path("outputs/training_results.json")
if _results_json.is_file():
    _saved = json.loads(_results_json.read_text())
    results = {
        "full_finetune": _saved.get("vit_full", full_best_acc),
        "partial_finetune": _saved.get("vit_partial", partial_best_acc),
        "lora_peft": _saved.get("vit_lora", lora_best_acc),
    }
else:
    results = {
        "full_finetune": full_best_acc,
        "partial_finetune": partial_best_acc,
        "lora_peft": lora_best_acc,
    }
best_strategy = max(results, key=results.get)
best_acc = results[best_strategy]
checkpoint_map = {
    "full_finetune": CHECKPOINT_DIR / "vit_full_finetune.pth",
    "partial_finetune": CHECKPOINT_DIR / "vit_partial_finetune.pth",
    "lora_peft": CHECKPOINT_DIR / "vit_lora_peft.pth",
}

print("=" * 60)
print("Best Performance Model")
print(f"Strategy: {best_strategy}")
print(f"Best test accuracy: {best_acc:.4f}")
print("=" * 60)

# Save a dedicated best checkpoint copy for submission clarity
best_ckpt_src = checkpoint_map[best_strategy]
best_ckpt_dst = CHECKPOINT_DIR / "vit_best_model.pth"
best_payload = torch.load(best_ckpt_src, map_location="cpu")
best_payload["best_strategy"] = best_strategy
torch.save(best_payload, best_ckpt_dst)

if best_strategy == "full_finetune":
    best_eval_builder = lambda: configure_full_finetune(build_vit_model())
elif best_strategy == "partial_finetune":
    best_eval_builder = lambda: configure_partial_finetune(build_vit_model(), num_last_layers=2)
else:
    best_eval_builder = lambda: apply_lora_to_vit(build_vit_model(), rank=8, alpha=16.0)

evaluate_vit_checkpoint(best_eval_builder, best_ckpt_dst, best_strategy)


# Task 4: Explore and Implement a Parameter-Efficient Fine-Tuning (PEFT) Technique (3 marks)

**This task is implemented in the code cell above** (same cell as Task 3 and Best Model). This heading is from the assignment template.

Summary:
- Method: **LoRA** (Low-Rank Adaptation, Hu et al. ICLR 2022)
- Where: `attention.qkv` in each Transformer block (`LoRALinear` + `apply_lora_to_vit`)
- Hyperparams: `rank=8`, `alpha=16.0`; freeze backbone, train LoRA + classifier head only
- Weights: `checkpoints/vit_lora_peft.pth` (skip training if present)

Scroll up to the **Task 4 - LoRA PEFT** output; if trained, you should see `Found checkpoint, skipping LoRA training.`


## Best Performance Model

The best model is selected by comparing the test accuracies of full fine-tuning, partial fine-tuning, and LoRA PEFT. The best checkpoint is saved as `checkpoints/vit_best_model.pth`.
